# 🏏 Does IPL Toss Really Matter? A Statistical Deep Dive

**Hypothesis:** Winning the toss gives a statistically significant advantage in IPL matches — but only in specific stadiums.

**Dataset:** IPL Matches (2008–2024) — `matches.csv`  
**Method:** Chi-Square Hypothesis Testing, Venue-level analysis  
**Goal:** Move beyond opinion and use statistics to determine whether toss outcomes have a real, measurable impact on match results.

## Step 1: Import Libraries

In [1]:
# Core data manipulation and numerical computing libraries
import pandas as pd   # For loading, filtering, and grouping the IPL dataset
import numpy as np    # For numerical operations (used in aggregations)
import matplotlib.pyplot as plt  # For plotting charts and visualizations

## Step 2: Load the Dataset

In [2]:
# Load the IPL matches dataset from a CSV file into a pandas DataFrame.
# The dataset contains match-level data including teams, toss details,
# venues, and match results across multiple IPL seasons.
df = pd.read_csv('matches.csv')

## Step 3: Initial Exploration — Preview the Raw Data

In [3]:
# Display the first 5 rows of the dataset to understand its structure.
# Key columns we care about:
#   - toss_winner   : which team won the toss
#   - toss_decision : did they choose to bat or field?
#   - winner        : which team ultimately won the match
#   - venue         : the stadium where the match was played
#   - result        : 'normal', 'tie', or 'no result'
#   - dl_applied    : 1 if Duckworth-Lewis method was applied (rain-affected match)
df.head()

,id,Season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,IPL-2017,Hyderabad,05-04-2017,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,2,IPL-2017,Pune,06-04-2017,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,normal,0,Rising Pune Supergiant,0,7,SPD Smith,Maharashtra Cricket Association Stadium,A Nand Kishore,S Ravi,NaN
2,3,IPL-2017,Rajkot,07-04-2017,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Kolkata Knight Riders,0,10,CA Lynn,Saurashtra Cricket Association Stadium,Nitin Menon,CK Nandan,NaN
3,4,IPL-2017,Indore,08-04-2017,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,normal,0,Kings XI Punjab,0,6,GJ Maxwell,Holkar Cricket Stadium,AK Chaudhary,C Shamshuddin,NaN
4,5,IPL-2017,Bangalore,08-04-2017,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,normal,0,Royal Challengers Bangalore,15,0,KM Jadhav,M Chinnaswamy Stadium,NaN,NaN,NaN


## Step 4: Select Only Relevant Columns

In [4]:
# The raw dataset has 18 columns, many of which (umpire names, player of match, etc.)
# are not needed for our statistical analysis.
# We keep only the 10 columns relevant to testing the toss-win hypothesis.
df = df[['Season', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'venue', 'result', 'dl_applied']]
df.head()

,Season,date,team1,team2,toss_winner,toss_decision,winner,venue,result,dl_applied
0,IPL-2017,05-04-2017,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Sunrisers Hyderabad,"Rajiv Gandhi International Stadium, Uppal",normal,0
1,IPL-2017,06-04-2017,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,Rising Pune Supergiant,Maharashtra Cricket Association Stadium,normal,0
2,IPL-2017,07-04-2017,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,Kolkata Knight Riders,Saurashtra Cricket Association Stadium,normal,0
3,IPL-2017,08-04-2017,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,Kings XI Punjab,Holkar Cricket Stadium,normal,0
4,IPL-2017,08-04-2017,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,Royal Challengers Bangalore,M Chinnaswamy Stadium,normal,0


## Step 5: Check Dataset Size Before Cleaning

In [5]:
# Check the total number of rows and columns in the dataset.
# Output: (756, 10) — 756 matches and 10 columns.
# This is our starting point BEFORE we remove incomplete or invalid matches.
df.shape

(756, 10)

## Step 6: Data Cleaning — Remove Invalid Matches

For hypothesis testing, we only want matches that had a **clear, fair result**:
- **No-result / ties** are excluded — there is no winner to analyze.
- **DL-affected matches** are excluded — rain disrupted the game, making it an unfair comparison since the toss advantage can be masked or exaggerated by weather conditions.

In [6]:
# Remove matches with no result or tie
# 'result' == 'normal' means the match was completed under standard conditions
df = df[df['result'] == 'normal']

# Remove DL applied matches (rain affected)
# When D/L is applied, the match result can be influenced by weather interruptions,
# making it difficult to attribute the outcome purely to the toss decision.
df = df[df['dl_applied'] == 0]

## Step 7: Check Dataset Size After Cleaning

In [7]:
# After removing ties, no-results, and rain-affected matches,
# we are left with 724 valid, completed matches.
# These 724 matches form the basis of all our statistical tests going forward.
df.shape

(724, 10)

## Step 8: Create the Target Variable — Did the Toss Winner Also Win the Match?

In [8]:
# Create a new binary column: 'toss_win_match_win'
#   - 1 = the team that won the toss ALSO won the match
#   - 0 = the team that won the toss LOST the match
#
# This is our core dependent variable — it directly captures
# whether the toss outcome translated into a match win.
# We compare toss_winner with winner row by row.
df['toss_win_match_win'] = (df['toss_winner'] == df['winner']).astype(int)

## Step 9: Calculate Overall Toss Win Rate

In [9]:
# Calculate the percentage of matches where the toss winner also won the game.
# The mean of a binary column gives the proportion of 1s (toss winner also won).
#
# Result: ~52.21% — slightly above the 50% random baseline.
# But is this margin statistically significant or just random noise?
# We will formally test this with Chi-Square in a later cell.
toss_win_rate = df['toss_win_match_win'].mean() * 100
print(f"Toss win rate: {toss_win_rate: .2f}%")

Toss win rate:  52.21%


## Step 10: Venue-Level Toss Win Rate (All Venues)

In [10]:
# Group matches by venue and compute:
#   - count : total matches played at that venue
#   - mean  : proportion of matches where the toss winner also won
# Then convert to percentage for readability.
#
# This reveals whether toss advantage varies by stadium —
# some grounds may have pitch or weather conditions that strongly
# favor batting first or chasing, making the toss more decisive.
venue_stats = df.groupby('venue')['toss_win_match_win'].agg(['count', 'mean']).reset_index()

venue_stats['toss_win_%'] = venue_stats['mean'] * 100

# Display top 10 venues where toss winners win the most often.
# Note: Many of these top-ranked venues have very few matches, making the % unreliable.
print(venue_stats.sort_values(by='toss_win_%', ascending=False).head(10))

                                      venue  count      mean  toss_win_%
0                          ACA-VDCA Stadium      2  1.000000  100.000000
25                          OUTsurance Oval      2  1.000000  100.000000
11                               Green Park      4  1.000000  100.000000
13                   Holkar Cricket Stadium      9  0.777778   77.777778
1                          Barabati Stadium      7  0.714286   71.428571
14                        IS Bindra Stadium      7  0.714286   71.428571
3                              Buffalo Park      3  0.666667   66.666667
4                     De Beers Diamond Oval      3  0.666667   66.666667
21  Maharashtra Cricket Association Stadium     21  0.666667   66.666667
35                     Sheikh Zayed Stadium      6  0.666667   66.666667


## Step 11: Filter to High-Volume Venues (30+ Matches)

Small sample sizes (e.g., 2–5 matches at a venue) can produce misleading statistics like a 100% toss-win rate. We restrict our analysis to venues with at least 30 matches for statistically reliable comparisons.

In [11]:
# Keep only venues with 30 or more matches played.
# This 30-match threshold ensures our percentages are based on
# enough data to draw meaningful conclusions.
#
# Result: 8 major IPL venues pass this filter, including
# Eden Gardens, Wankhede, Feroz Shah Kotla, Chepauk, and others.
venue_stats_filtered = venue_stats[venue_stats['count'] >= 30]
print(venue_stats_filtered.sort_values(by='toss_win_%', ascending=False))

                                         venue  count      mean  toss_win_%
8                                 Eden Gardens     73  0.575342   57.534247
17                       M Chinnaswamy Stadium     67  0.552239   55.223881
32                      Sawai Mansingh Stadium     46  0.543478   54.347826
20             MA Chidambaram Stadium, Chepauk     48  0.541667   54.166667
9                             Feroz Shah Kotla     63  0.507937   50.793651
40                            Wankhede Stadium     72  0.500000   50.000000
27  Punjab Cricket Association Stadium, Mohali     35  0.457143   45.714286
28   Rajiv Gandhi International Stadium, Uppal     53  0.339623   33.962264


## Step 12: Overall Value Counts — Wins vs Losses for Toss Winners

In [12]:
# Count how many times the toss winner won (1) vs lost (0) across all 724 matches.
# Result:
#   1 → 378 matches : toss winner also won the match
#   0 → 346 matches : toss winner lost the match
#
# These observed counts are the direct input to our Chi-Square test.
df['toss_win_match_win'].value_counts()

toss_win_match_win
1    378
0    346
Name: count, dtype: int64

## Step 13: Hypothesis Test 1 — Chi-Square Test (Overall Toss Advantage)

**H₀ (Null):** Toss winners win matches at exactly 50% — same as random chance.  
**H₁ (Alternative):** Toss winners win at a rate significantly different from 50%.  
**Significance level:** α = 0.05 (we need p < 0.05 to reject H₀)

In [13]:
from scipy.stats import chisquare

# Observed: actual win/loss counts for toss winners across 724 matches
observed = [378, 346]

# Expected: if the toss has NO effect, we'd expect a perfect 50/50 split
# Total = 724 → 362 wins and 362 losses expected under H₀
expected = [362, 362]

# Chi-Square test measures how far the observed distribution deviates from expected.
# A large chi-square statistic (small p-value) means the difference is unlikely by chance.
chi_stat, p_value = chisquare(f_obs=observed, f_exp=expected)

print("Chi-square:", chi_stat)
print("p-value:", p_value)

# RESULT: p-value = 0.234 > 0.05
# CONCLUSION: We FAIL to reject H₀.
# The 52.2% toss-win rate is NOT statistically significant overall.
# The slight edge could easily be due to random variation across 724 matches.

Chi-square: 1.4143646408839778
p-value: 0.23433318720620627


## Step 14: Venue-Specific Test — Eden Gardens (Kolkata)

In [14]:
# Filter the dataset to only matches played at Eden Gardens.
# Eden Gardens (Kolkata) is one of the most iconic cricket grounds in the world.
# We want to test whether the toss carries a stronger advantage at this specific venue.
eden_df = df[df['venue'] == 'Eden Gardens']

# Toss win/loss count at Eden Gardens:
#   1 → 42 matches : toss winner won
#   0 → 31 matches : toss winner lost
# This gives a 57.5% toss-win rate — higher than the overall average.
eden_df['toss_win_match_win'].value_counts()

toss_win_match_win
1    42
0    31
Name: count, dtype: int64

## Step 15: Chi-Square Test at Eden Gardens

In [15]:
from scipy.stats import chisquare

# At Eden Gardens: 42 wins vs 31 losses for toss winners
# Under H₀ (no toss effect): we expect 36.5 wins and 36.5 losses (50/50 of 73 total)
observed = [42, 31]
expected = [36.5, 36.5]

chi_stat, p_value = chisquare(f_obs=observed, f_exp=expected)

print("Chi-square:", chi_stat)
print("p-value:", p_value)

# RESULT: p-value = 0.198 > 0.05
# CONCLUSION: NOT statistically significant even at Eden Gardens.
# Despite the 57.5% win rate, the sample size of 73 matches is not
# sufficient to rule out random variation with confidence.

Chi-square: 1.6575342465753424
p-value: 0.19793657403507028


## Step 16: Chi-Square Test Across ALL Major Venues (Automated Loop)

Instead of testing one venue at a time manually, we **loop through all venues with 30+ matches** and run a Chi-Square test for each. This gives a complete, scalable picture of where toss advantage is statistically significant.

In [16]:
from scipy.stats import chisquare

# List to store Chi-Square test results for each venue
results = []

# Loop through every unique venue in the cleaned dataset
for venue in df['venue'].unique():
    
    temp_df = df[df['venue'] == venue]
    
    # Skip venues with fewer than 30 matches — too small for reliable statistics
    if len(temp_df) < 30:
        continue
    
    # Count how many times the toss winner won and lost at this venue
    counts = temp_df['toss_win_match_win'].value_counts()
    
    wins = counts.get(1, 0)    # Toss winner also won the match
    losses = counts.get(0, 0)  # Toss winner lost the match
    
    total = wins + losses
    
    # Under H₀: expected is a perfect 50/50 split at every venue
    expected = [total/2, total/2]
    observed = [wins, losses]
    
    # Run Chi-Square test for this specific venue
    chi_stat, p_value = chisquare(f_obs=observed, f_exp=expected)
    
    toss_win_pct = (wins / total) * 100
    
    # Store this venue's results including a significance flag
    results.append({
        'venue': venue,
        'matches': total,
        'toss_win_%': toss_win_pct,
        'p_value': p_value,
        'significant': p_value < 0.05  # True if the toss effect is statistically real at this venue
    })

results_df = pd.DataFrame(results)

# Sort by p-value ascending — venues with strongest toss effect appear first
print(results_df.sort_values(by='p_value'))

# KEY FINDING:
# Rajiv Gandhi International Stadium (Uppal, Hyderabad) is the ONLY major venue
# where the toss effect is statistically significant (p = 0.020 < 0.05).
#
# Counterintuitively, toss winners there win only ~34% of the time —
# meaning the team that LOSES the toss has a significant advantage at this ground!
# All other major IPL venues show NO statistically significant toss effect.

                                        venue  matches  toss_win_%   p_value  \
0   Rajiv Gandhi International Stadium, Uppal       53   33.962264  0.019537   
3                                Eden Gardens       73   57.534247  0.197937   
1                       M Chinnaswamy Stadium       67   55.223881  0.392448   
6                      Sawai Mansingh Stadium       46   54.347826  0.555346   
7             MA Chidambaram Stadium, Chepauk       48   54.166667  0.563703   
5  Punjab Cricket Association Stadium, Mohali       35   45.714286  0.612090   
4                            Feroz Shah Kotla       63   50.793651  0.899741   
2                            Wankhede Stadium       72   50.000000  1.000000   

   significant  
0         True  
3        False  
1        False  
6        False  
7        False  
5        False  
4        False  
2        False  


## 📊 Summary of Findings

| Test | Venue | Toss Win % | p-value | Significant? |
|------|-------|-----------|---------|-------------|
| Overall | All 724 matches | 52.2% | 0.234 | ❌ No |
| Venue-specific | Eden Gardens | 57.5% | 0.198 | ❌ No |
| Venue-specific | Wankhede Stadium | 50.0% | 1.000 | ❌ No |
| Venue-specific | Rajiv Gandhi Stadium (Uppal) | 34.0% | **0.020** | ✅ Yes |

### 🎯 Conclusion
Winning the toss does **not** provide a statistically significant match advantage at most IPL venues. The one notable exception is the Rajiv Gandhi International Stadium (Uppal), where toss **losers** actually win significantly more often — a counterintuitive and interesting finding that warrants deeper investigation (e.g., pitch behavior, team strategies at Hyderabad).
